# Module 2 - Final Integration: Classification vs Regression

This notebook is the **final integration** step for Module 2. It does **not** modify
`01_eda.ipynb`, `02_classification.ipynb`, `03_regression.ipynb`, or Module 1.

It:
1. Reads only the CSVs already produced by those notebooks from `analytics/outputs/`.
2. Builds a combined `model_comparison.csv` (classification vs regression).
3. Reports the imbalance comparison and the GridSearchCV Random Forest results.
4. Writes an evidence-based `deployment_recommendation.txt`.
5. Rebuilds and persists the confirmed classification `Pipeline` (Logistic Regression)
   with `joblib`, reloads it, and tests it on **raw, unprocessed** input.

In [1]:
import os, json, ast
from pathlib import Path
if Path.cwd().name == "analytics":
    os.chdir("..")

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

RANDOM_STATE = 42
OUT = Path("analytics/outputs")
OUT.mkdir(parents=True, exist_ok=True)
print("cwd:", Path.cwd(), "| outputs dir:", OUT)

cwd: C:\Users\Lenovo\Desktop\zepto-data-ai-platform | outputs dir: analytics\outputs


## 1. Load the existing result CSVs

These are the actual results already produced by `02_classification.ipynb` and
`03_regression.ipynb`. No metrics are recomputed.

In [2]:
classification_df = pd.read_csv(OUT / "classification_comparison.csv")
imbalance_df = pd.read_csv(OUT / "imbalance_comparison.csv")
tuned_rf_df = pd.read_csv(OUT / "tuned_random_forest.csv")
regression_df = pd.read_csv(OUT / "regression_results.csv")

print("classification_comparison.csv:")
display(classification_df)
print()
print("imbalance_comparison.csv:")
display(imbalance_df)
print()
print("tuned_random_forest.csv:")
display(tuned_rf_df)
print()
print("regression_results.csv:")
display(regression_df)

classification_comparison.csv:


,model,accuracy,precision,recall,f1,roc_auc
0,Logistic Regression,0.825843,0.813559,0.705882,0.755906,0.858422
1,Decision Tree,0.780899,0.754386,0.632353,0.688000,0.826471
2,Random Forest,0.780899,0.723077,0.691176,0.706767,0.835495



imbalance_comparison.csv:


,version,precision,recall,f1,roc_auc,accuracy
0,A_baseline,0.813559,0.705882,0.755906,0.858422,0.825843
1,B_balanced,0.712329,0.764706,0.737589,0.859225,0.792135
2,C_SMOTE_train_only,0.728571,0.750000,0.739130,0.862032,0.797753



tuned_random_forest.csv:


,best_params,best_cv_f1,oob_score,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc
0,"{'clf__max_depth': 10, 'clf__max_features': 0....",0.7549,0.82,0.8202,0.7903,0.7206,0.7538,0.8417



regression_results.csv:


,metric,value,n_test,p_predictors
0,"['MAE', 'RMSE', 'R2', 'Adjusted_R2']","[17.688668575875703, 43.32576824816406, 0.3641...",178,15


## 2. Classification-vs-regression comparison -> model_comparison.csv

Each row is tagged with a `task` (classification / regression). Classification rows
carry accuracy/precision/recall/f1/roc_auc; the regression row carries MAE/RMSE/R2/Adjusted_R2.
Values are read directly from the CSVs above (no recalculation).

In [3]:
reg_row = regression_df.iloc[0]
reg_metrics = ast.literal_eval(reg_row["metric"])
reg_values = ast.literal_eval(reg_row["value"])
reg_dict = dict(zip(reg_metrics, reg_values))
print("Parsed regression metrics:", reg_dict)

metric_cols = ["accuracy", "precision", "recall", "f1", "roc_auc", "MAE", "RMSE", "R2", "Adjusted_R2"]

rows = []
for _, r in classification_df.iterrows():
    row = {"task": "classification", "model": r["model"]}
    for m in ["accuracy", "precision", "recall", "f1", "roc_auc"]:
        row[m] = r[m]
    for m in ["MAE", "RMSE", "R2", "Adjusted_R2"]:
        row[m] = np.nan
    rows.append(row)

t = tuned_rf_df.iloc[0]
rows.append({
    "task": "classification", "model": "Tuned Random Forest",
    "accuracy": t["test_accuracy"], "precision": t["test_precision"],
    "recall": t["test_recall"], "f1": t["test_f1"], "roc_auc": t["test_roc_auc"],
    "MAE": np.nan, "RMSE": np.nan, "R2": np.nan, "Adjusted_R2": np.nan,
})

rows.append({
    "task": "regression", "model": "Linear Regression",
    "accuracy": np.nan, "precision": np.nan, "recall": np.nan, "f1": np.nan, "roc_auc": np.nan,
    "MAE": reg_dict["MAE"], "RMSE": reg_dict["RMSE"],
    "R2": reg_dict["R2"], "Adjusted_R2": reg_dict["Adjusted_R2"],
})

model_comparison = pd.DataFrame(rows)
model_comparison = model_comparison[["task", "model"] + metric_cols]
model_comparison.to_csv(OUT / "model_comparison.csv", index=False)
display(model_comparison)

Parsed regression metrics: {'MAE': 17.688668575875703, 'RMSE': 43.32576824816406, 'R2': 0.36416178055053894, 'Adjusted_R2': 0.30528787134225555}


,task,model,accuracy,precision,recall,f1,roc_auc,MAE,RMSE,R2,Adjusted_R2
0,classification,Logistic Regression,0.825843,0.813559,0.705882,0.755906,0.858422,NaN,NaN,NaN,NaN
1,classification,Decision Tree,0.780899,0.754386,0.632353,0.688000,0.826471,NaN,NaN,NaN,NaN
2,classification,Random Forest,0.780899,0.723077,0.691176,0.706767,0.835495,NaN,NaN,NaN,NaN
3,classification,Tuned Random Forest,0.820200,0.790300,0.720600,0.753800,0.841700,NaN,NaN,NaN,NaN
4,regression,Linear Regression,NaN,NaN,NaN,NaN,NaN,17.688669,43.325768,0.364162,0.305288


## 3. Imbalance comparison (Logistic Regression)

Baseline vs class_weight="balanced" vs SMOTE applied **only to the training fold**.

In [4]:
imbalance_out = imbalance_df[["version", "precision", "recall", "f1", "roc_auc", "accuracy"]]
display(imbalance_out)

base = imbalance_df[imbalance_df["version"] == "A_baseline"].iloc[0]
smote = imbalance_df[imbalance_df["version"] == "C_SMOTE_train_only"].iloc[0]
print(f"Baseline  -> precision={base['precision']:.4f} recall={base['recall']:.4f} f1={base['f1']:.4f} auc={base['roc_auc']:.4f}")
print(f"SMOTE     -> precision={smote['precision']:.4f} recall={smote['recall']:.4f} f1={smote['f1']:.4f} auc={smote['roc_auc']:.4f}")
print(f"Recall gain from baseline -> SMOTE: {smote['recall'] - base['recall']:+.4f}")
print(f"AUC gain from baseline -> SMOTE: {smote['roc_auc'] - base['roc_auc']:+.4f}")

,version,precision,recall,f1,roc_auc,accuracy
0,A_baseline,0.813559,0.705882,0.755906,0.858422,0.825843
1,B_balanced,0.712329,0.764706,0.737589,0.859225,0.792135
2,C_SMOTE_train_only,0.728571,0.750000,0.739130,0.862032,0.797753


Baseline  -> precision=0.8136 recall=0.7059 f1=0.7559 auc=0.8584
SMOTE     -> precision=0.7286 recall=0.7500 f1=0.7391 auc=0.8620
Recall gain from baseline -> SMOTE: +0.0441
AUC gain from baseline -> SMOTE: +0.0036


## 4. GridSearchCV Random Forest results

Best parameters, best CV F1, OOB score, and test metrics (all from the existing CSV).

In [5]:
g = tuned_rf_df.iloc[0]
print("Best parameters:", g["best_params"])
print("Best CV F1:", round(float(g["best_cv_f1"]), 4))
print("OOB score:", round(float(g["oob_score"]), 4))
print("Test accuracy:", round(float(g["test_accuracy"]), 4))
print("Test precision:", round(float(g["test_precision"]), 4))
print("Test recall:", round(float(g["test_recall"]), 4))
print("Test F1:", round(float(g["test_f1"]), 4))
print("Test ROC-AUC:", round(float(g["test_roc_auc"]), 4))

print()
print("Comparison of base vs tuned Random Forest (test F1):")
rf_base_f1 = classification_df.loc[classification_df["model"] == "Random Forest", "f1"].iloc[0]
print(f"  base RF F1 = {rf_base_f1:.4f}")
print(f"  tuned RF F1 = {float(g['test_f1']):.4f}")
print(f"  CV F1      = {float(g['best_cv_f1']):.4f}")

Best parameters: {'clf__max_depth': 10, 'clf__max_features': 0.5, 'clf__n_estimators': 200}
Best CV F1: 0.7549
OOB score: 0.82
Test accuracy: 0.8202
Test precision: 0.7903
Test recall: 0.7206
Test F1: 0.7538
Test ROC-AUC: 0.8417

Comparison of base vs tuned Random Forest (test F1):
  base RF F1 = 0.7068
  tuned RF F1 = 0.7538
  CV F1      = 0.7549


## 5. Deployment recommendation

Generated **only** from the actual classifier metrics already produced. No values are invented.

In [6]:
lr = classification_df[classification_df["model"] == "Logistic Regression"].iloc[0]
rf_base = classification_df[classification_df["model"] == "Random Forest"].iloc[0]
dt = classification_df[classification_df["model"] == "Decision Tree"].iloc[0]
t = tuned_rf_df.iloc[0]
smote = imbalance_df[imbalance_df["version"] == "C_SMOTE_train_only"].iloc[0]

rec = "\n".join([
    "1. Deploy the Logistic Regression pipeline as the primary survival-prediction model.",
    f"   It is the strongest base model on every ranking metric: F1={lr['f1']:.4f}, ROC-AUC={lr['roc_auc']:.4f}, "
    f"and the highest precision={lr['precision']:.4f}, while being simple and fast to score.",
    f"2. The tuned Random Forest does not improve on the base Random Forest and does not beat Logistic Regression "
    f"(tuned F1={float(t['test_f1']):.4f}, ROC-AUC={float(t['test_roc_auc']):.4f}, CV F1={float(t['best_cv_f1']):.4f}, OOB={float(t['oob_score']):.4f}), "
    "so its added complexity and cost are not justified by these results.",
    f"3. The depth-limited Decision Tree underperforms (F1={dt['f1']:.4f}, recall={dt['recall']:.4f}), so do not ship it as primary.",
    f"4. If the operational priority is maximising detection of survivors (recall), the SMOTE-on-train-only variant "
    f"lifts recall to {smote['recall']:.4f} (from {lr['recall']:.4f}) with the best ROC-AUC={smote['roc_auc']:.4f}, at a small precision cost ({smote['precision']:.4f}); otherwise use the default Logistic Regression pipeline.",
    "5. The dataset is small (178 test rows), so these metrics should be re-validated on a larger sample before production.",
])

(OUT / "deployment_recommendation.txt").write_text(rec + "\n")
print(rec)

1. Deploy the Logistic Regression pipeline as the primary survival-prediction model.
   It is the strongest base model on every ranking metric: F1=0.7559, ROC-AUC=0.8584, and the highest precision=0.8136, while being simple and fast to score.
2. The tuned Random Forest does not improve on the base Random Forest and does not beat Logistic Regression (tuned F1=0.7538, ROC-AUC=0.8417, CV F1=0.7549, OOB=0.8200), so its added complexity and cost are not justified by these results.
3. The depth-limited Decision Tree underperforms (F1=0.6880, recall=0.6324), so do not ship it as primary.
4. If the operational priority is maximising detection of survivors (recall), the SMOTE-on-train-only variant lifts recall to 0.7500 (from 0.7059) with the best ROC-AUC=0.8620, at a small precision cost (0.7286); otherwise use the default Logistic Regression pipeline.
5. The dataset is small (178 test rows), so these metrics should be re-validated on a larger sample before production.


## 6. Rebuild and persist the confirmed classification Pipeline

The persistence uses the **exact** classification methodology already confirmed in
`02_classification.ipynb`:

- `feature_cols = ["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked", "deck"]`
- `numeric_features = ["age", "fare", "sibsp", "parch"]`
- `categorical_features = ["sex", "pclass", "embarked", "deck"]`
- `make_preprocessor()`: numeric -> SimpleImputer(median) -> StandardScaler;
  categorical -> SimpleImputer(most_frequent) -> OneHotEncoder(handle_unknown="ignore", sparse_output=False)
- `Pipeline([("pre", make_preprocessor()), ("clf", classifier)])`

Fitted with the LogisticRegression classifier:
`LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)`

In [7]:
df = pd.read_csv("analytics/titanic.csv")

cleaned = df.copy()
cleaned = cleaned.dropna(subset=["embarked", "embark_town"])
group_med = cleaned.groupby(["sex", "pclass"])["age"].transform("median")
cleaned["age"] = cleaned["age"].fillna(group_med)
cleaned["age"] = cleaned["age"].fillna(cleaned["age"].median())
cleaned["deck"] = cleaned["deck"].fillna("Unknown")
cleaned["pclass"] = cleaned["pclass"].astype(str)

feature_cols = ["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked", "deck"]
X = cleaned[feature_cols]
y = cleaned["survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

numeric_features = ["age", "fare", "sibsp", "parch"]
categorical_features = ["sex", "pclass", "embarked", "deck"]


def make_preprocessor():
    numeric = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    return ColumnTransformer([
        ("num", numeric, numeric_features),
        ("cat", categorical, categorical_features),
    ])


def make_pipeline(clf):
    return Pipeline([("pre", make_preprocessor()), ("clf", clf)])


clf_pipeline = make_pipeline(LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
clf_pipeline.fit(X_train, y_train)

joblib.dump(clf_pipeline, OUT / "titanic_classification_pipeline.joblib")
print("Saved pipeline to:", OUT / "titanic_classification_pipeline.joblib")
print("Pipeline type:", type(clf_pipeline))
print("Steps:", [name for name, _ in clf_pipeline.steps])
print("Train accuracy:", round(clf_pipeline.score(X_train, y_train), 4))
print("Test  accuracy:", round(clf_pipeline.score(X_test, y_test), 4))

Saved pipeline to:

 analytics\outputs\titanic_classification_pipeline.joblib
Pipeline type: <class 'sklearn.pipeline.Pipeline'>
Steps: ['pre', 'clf']
Train accuracy: 0.8158
Test  accuracy: 0.8258


## 7. Reload the persisted pipeline and test on RAW, UNPRECESSED input

The input below is raw feature rows with the original 8 columns. No manual imputation,
scaling, or encoding is applied -- the loaded `Pipeline` performs all preprocessing
internally. One row deliberately contains a missing `age` and a 'Unknown' deck to exercise
the imputer and one-hot encoder, and `pclass` is supplied as a string matching the cleaned
training schema.

In [8]:
loaded = joblib.load(OUT / "titanic_classification_pipeline.joblib")
print("Reloaded pipeline type:", type(loaded))
print("Reloaded steps:", [name for name, _ in loaded.steps])

raw_test = pd.DataFrame([
    {"pclass": "3", "sex": "male", "age": np.nan, "sibsp": 1,
     "parch": 0, "fare": 7.25, "embarked": "S", "deck": "Unknown"},
    {"pclass": "1", "sex": "female", "age": 38.0, "sibsp": 0,
     "parch": 0, "fare": 85.0, "embarked": "S", "deck": "C"},
])

print()
print("Raw input shape:", raw_test.shape)
print("Raw input columns:", list(raw_test.columns))
display(raw_test)

pred = loaded.predict(raw_test)
pred_proba = loaded.predict_proba(raw_test)

print()
print("Prediction:", pred.tolist())
print("Prediction probability:")
display(pd.DataFrame(pred_proba, columns=[f"P({c})" for c in loaded.classes_]))
print("Classes:", loaded.classes_.tolist())

assert len(pred) == raw_test.shape[0]
assert pred_proba.shape == (raw_test.shape[0], len(loaded.classes_))
print()
print("Verification OK: pipeline produced a prediction for every raw row.")

Reloaded pipeline type: <class 'sklearn.pipeline.Pipeline'>
Reloaded steps: ['pre', 'clf']

Raw input shape: (2, 8)
Raw input columns: ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'deck']


,pclass,sex,age,sibsp,parch,fare,embarked,deck
0,3,male,NaN,1,0,7.25,S,Unknown
1,1,female,38.0,0,0,85.00,S,C


Prediction:

 [0, 1]
Prediction probability:


,P(0),P(1)
0,0.923517,0.076483
1,0.093248,0.906752


Classes: [0, 1]

Verification OK: pipeline produced a prediction for every raw row.


## 8. Verify all expected output files exist

In [9]:
expected = [
    "analytics/outputs/classification_comparison.csv",
    "analytics/outputs/imbalance_comparison.csv",
    "analytics/outputs/tuned_random_forest.csv",
    "analytics/outputs/regression_results.csv",
    "analytics/outputs/model_comparison.csv",
    "analytics/outputs/deployment_recommendation.txt",
    "analytics/outputs/titanic_classification_pipeline.joblib",
]
missing = []
for f in expected:
    p = Path(f)
    ok = p.exists()
    print(f"{f}: {'OK' if ok else 'MISSING'}")
    if not ok:
        missing.append(f)

if missing:
    raise FileNotFoundError("Missing expected outputs: " + str(missing))
print()
print("All expected output files present.")

analytics/outputs/classification_comparison.csv: OK
analytics/outputs/imbalance_comparison.csv: OK
analytics/outputs/tuned_random_forest.csv: OK
analytics/outputs/regression_results.csv: OK
analytics/outputs/model_comparison.csv: OK
analytics/outputs/deployment_recommendation.txt: OK
analytics/outputs/titanic_classification_pipeline.joblib: OK

All expected output files present.
